<a href="https://colab.research.google.com/github/jt7827/senior-thesis-code/blob/main/Thesis_MH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os
import pandas as pd

# Works when cloned from GitHub or opened in Colab from GitHub
local_csv = "data/archive (1).csv"
local_zip = "data/archive (1).zip"

if os.path.exists(local_csv):
    df = pd.read_csv(local_csv)
elif os.path.exists(local_zip):
    df = pd.read_csv(local_zip)
else:
    # Fallback to your Google Drive if running privately in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    df = pd.read_csv('/content/drive/MyDrive/MH Data/Combined Data.csv')

print("Dataset size:", df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset size: (53043, 3)


,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [16]:
# Record the original dataset size
original_rows = len(df)

print("Original rows:", original_rows)
print("\nMissing values before cleaning:")
print(df.isnull().sum())

# Drop rows with missing values for text or label
df = df.dropna(subset=["statement", "status"]).copy()

print("\nDataset size after dropping null values:", df.shape)
print("Rows removed:", original_rows - len(df))
print("Remaining missing values:", df.isnull().sum().sum())

Original rows: 53043

Missing values before cleaning:
Unnamed: 0      0
statement     362
status          0
dtype: int64

Dataset size after dropping null values: (52681, 3)
Rows removed: 362
Remaining missing values: 0


In [17]:
print("Total rows after cleaning:", len(df))
print("Total columns:", len(df.columns))

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Total rows after cleaning: 52681
Total columns: 3

Missing values after cleaning:
Unnamed: 0    0
statement     0
status        0
dtype: int64

Duplicate rows: 0


In [18]:
label_col = "status"

label_counts = df[label_col].value_counts()

summary = pd.DataFrame({
    "Frequency": label_counts,
    "Percentage": (
        label_counts / label_counts.sum() * 100
    ).round(2)
})

display(summary)

,Frequency,Percentage
status,,
Normal,16343,31.02
Depression,15404,29.24
Suicidal,10652,20.22
Anxiety,3841,7.29
Bipolar,2777,5.27
Stress,2587,4.91
Personality disorder,1077,2.04


In [19]:
df["binary_label"] = df[label_col].apply(
    lambda x: "Normal"
    if str(x).strip().lower() == "normal"
    else "Abnormal"
)

binary_summary = (
    df["binary_label"]
    .value_counts()
    .to_frame("Frequency")
)

binary_summary["Percentage"] = (
    binary_summary["Frequency"]
    / binary_summary["Frequency"].sum()
    * 100
).round(2)

display(binary_summary)

,Frequency,Percentage
binary_label,,
Abnormal,36338,68.98
Normal,16343,31.02


In [20]:
text_col = "statement"

# Calculate text lengths
df["word_count"] = df[text_col].astype(str).str.split().str.len()
df["char_count"] = df[text_col].astype(str).str.len()

print("Word count summary:")
display(df.groupby(label_col)["word_count"].describe().round(2))

print("Character count summary:")
display(df.groupby(label_col)["char_count"].describe().round(2))

Word count summary:


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Anxiety,3841.0,143.84,152.63,1.0,41.0,102.0,192.0,1592.0
Bipolar,2777.0,176.20,176.49,4.0,74.0,129.0,220.0,4804.0
Depression,15404.0,168.02,188.23,1.0,54.0,113.0,215.0,4239.0
Normal,16343.0,17.25,22.77,1.0,6.0,10.0,18.0,255.0
Personality disorder,1077.0,179.35,221.89,5.0,66.0,137.0,236.0,5419.0
Stress,2587.0,114.59,105.90,1.0,64.0,87.0,127.0,1606.0
Suicidal,10652.0,146.44,186.96,1.0,42.0,92.0,186.0,6300.0


Character count summary:


,count,mean,std,min,25%,50%,75%,max
status,,,,,,,,
Anxiety,3841.0,764.68,812.38,7.0,225.0,544.0,1009.00,8539.0
Bipolar,2777.0,946.07,952.51,27.0,400.0,699.0,1173.00,26163.0
Depression,15404.0,844.03,953.00,3.0,272.0,562.0,1078.25,19822.0
Normal,16343.0,90.24,120.90,2.0,28.0,50.0,96.00,1358.0
Personality disorder,1077.0,956.73,1149.83,39.0,358.0,736.0,1249.00,27588.0
Stress,2587.0,613.55,570.07,6.0,336.5,459.0,672.50,8389.0
Suicidal,10652.0,734.97,982.21,6.0,203.0,453.0,923.25,32759.0


In [21]:
length_summary = (
    df.groupby(label_col)[["word_count", "char_count"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(length_summary)

print("Original rows:", original_rows)
print("Rows analyzed:", len(df))
print("Rows excluded during cleaning:", original_rows - len(df))



word_count                                  char_count  \
                          count    mean median     std min   max      count   
status                                                                        
Anxiety                    3841  143.84  102.0  152.63   1  1592       3841   
Bipolar                    2777  176.20  129.0  176.49   4  4804       2777   
Depression                15404  168.02  113.0  188.23   1  4239      15404   
Normal                    16343   17.25   10.0   22.77   1   255      16343   
Personality disorder       1077  179.35  137.0  221.89   5  5419       1077   
Stress                     2587  114.59   87.0  105.90   1  1606       2587   
Suicidal                  10652  146.44   92.0  186.96   1  6300      10652   

                                                         
                        mean median      std min    max  
status                                                   
Anxiety               764.68  544.0   812.38   7   8539  
Bipolar               946.07  699.0   952.51  27  26163  
Depression            844.03  562.0   953.00   3  19822  
Normal                 90.24   50.0   120.90   2   1358  
Personality disorder  956.73  736.0  1149.83  39  27588  
Stress                613.55  459.0   570.07   6   8389  
Suicidal              734.97  453.0   982.21   6  32759

Original rows: 53043
Rows analyzed: 52681
Rows excluded during cleaning: 362


Normal text is way wayyy smaller than other categories-exclude from multiclass classification

In [22]:
#start feature extraction

!pip -q install spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 24.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:

import pandas as pd
import numpy as np
import re
import spacy

# Start with statements and labels that are present
data = df.dropna(subset=["statement", "status"]).copy()
data["statement"] = data["statement"].astype(str)
data = data[data["statement"].str.strip().ne("")].copy()

nlp = spacy.load("en_core_web_sm")

# Define the linguistic categories
first_person = {
    "i", "me", "my", "mine", "myself"
}

absolutist_words = {
    "absolutely", "all", "always", "complete",
    "completely", "constant", "constantly", "definitely",
    "entire", "ever", "every", "everyone",
    "everything", "full", "must", "never", "nothing", "totally", "whole"
}

# Count exact word tokens, rather than matching substrings
def count_dictionary_words(tokens, dictionary):
    return sum(token.lower() in dictionary for token in tokens)

# Process statements in batches for efficiency
past_tense_counts = []
word_counts = []
first_person_counts = []
absolutist_counts = []

statements = data["statement"].tolist()

for doc in nlp.pipe(statements, batch_size=500):
    tokens = [
        token.text.lower()
        for token in doc
        if not token.is_space and not token.is_punct
    ]

    n_words = len(tokens)

    # spaCy VBD = simple past-tense verb
    n_past = sum(
        token.tag_ == "VBD"
        for token in doc
    )

    n_first_person = count_dictionary_words(
        tokens, first_person
    )

    n_absolutist = count_dictionary_words(
        tokens, absolutist_words
    )

    word_counts.append(n_words)
    past_tense_counts.append(n_past)
    first_person_counts.append(n_first_person)
    absolutist_counts.append(n_absolutist)

data["word_count"] = word_counts
data["past_tense_count"] = past_tense_counts
data["first_person_count"] = first_person_counts
data["absolutist_count"] = absolutist_counts

# Rates per 100 words; avoid division by zero
denominator = data["word_count"].replace(0, np.nan)

data["past_tense_rate"] = (
    data["past_tense_count"] / denominator * 100
)

data["first_person_singular_rate"] = (
    data["first_person_count"] / denominator * 100
)

data["absolutist_rate"] = (
    data["absolutist_count"] / denominator * 100
)

print("Statements analyzed:", len(data))
display(data[[
    "status",
    "word_count",
    "past_tense_rate",
    "first_person_singular_rate",
    "absolutist_rate"
]].head())


Statements analyzed: 52681


,status,word_count,past_tense_rate,first_person_singular_rate,absolutist_rate
0,Anxiety,3,0.0,33.333333,0.000000
1,Anxiety,10,0.0,0.000000,10.000000
2,Anxiety,14,0.0,0.000000,7.142857
3,Anxiety,13,0.0,23.076923,0.000000
4,Anxiety,16,0.0,6.250000,0.000000


In [11]:

# Linguistic features to summarize
features = [
    "past_tense_rate",
    "first_person_singular_rate",
    "absolutist_rate"
]

# Summary statistics by mental-health category
summary = (
    data.groupby("status")[features]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

display(summary)


past_tense_rate                     \
                               count  mean median   std   
status                                                    
Anxiety                         3841  2.92   2.15  3.12   
Bipolar                         2777  3.14   2.56  2.69   
Depression                     15404  2.44   1.69  2.78   
Normal                         16343  2.72   0.00  6.04   
Personality disorder            1077  2.53   1.71  2.67   
Stress                          2587  3.13   2.04  3.51   
Suicidal                       10652  2.25   1.41  2.75   

                     first_person_singular_rate                      \
                                          count   mean median   std   
status                                                                
Anxiety                                    3841   9.10   9.70  4.81   
Bipolar                                    2777   9.75  10.04  3.63   
Depression                                15404  10.88  11.45  4.94   
Normal                                    16343   5.44   0.00  7.72   
Personality disorder                       1077   9.28   9.96  4.47   
Stress                                     2587   9.32  10.00  4.57   
Suicidal                                  10652  12.14  12.40  5.19   

                     absolutist_rate                     
                               count  mean median   std  
status                                                   
Anxiety                         3841  1.27   0.87  1.92  
Bipolar                         2777  1.27   1.07  1.26  
Depression                     15404  1.68   1.37  1.80  
Normal                         16343  0.88   0.00  3.36  
Personality disorder            1077  1.40   1.18  1.39  
Stress                          2587  1.25   1.04  1.40  
Suicidal                       10652  1.74   1.38  2.00

In [12]:

from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

features = {
    "Past-tense verbs": "past_tense_rate",
    "First-person singular pronouns": "first_person_singular_rate",
    "Absolutist language": "absolutist_rate"
}

results = []

for feature_name, col in features.items():
    groups = [
        group[col].dropna().to_numpy()
        for _, group in data.groupby("status")
    ]
    groups = [g for g in groups if len(g) > 0]

    H, p = kruskal(*groups)

    # Epsilon-squared effect size for Kruskal-Wallis
    n = sum(len(g) for g in groups)
    k = len(groups)
    epsilon_sq = max(0, (H - k + 1) / (n - k))

    results.append({
        "Feature": feature_name,
        "Kruskal-Wallis H": H,
        "P-value": p,
        "Effect size (epsilon-squared)": epsilon_sq,
        "% statements with zero": (
            data[col].eq(0).mean() * 100
        )
    })

results = pd.DataFrame(results)

# Correct for testing multiple features
results["Adjusted p-value"] = multipletests(
    results["P-value"], method="fdr_bh"
)[1]

display(results.round(5))


,Feature,Kruskal-Wallis H,P-value,Effect size (epsilon-squared),% statements with zero,Adjusted p-value
0,Past-tense verbs,4017.87259,0.0,0.07616,43.94564,0.0
1,First-person singular pronouns,10170.83505,0.0,0.19298,21.50111,0.0
2,Absolutist language,10472.73299,0.0,0.19871,47.61299,0.0


In [13]:

results_display = results.copy()

# Display very small p-values in scientific notation
results_display["P-value"] = results_display["P-value"].map(
    lambda x: f"{x:.3e}"
)

results_display["Adjusted p-value"] = results_display[
    "Adjusted p-value"
].map(lambda x: f"{x:.3e}")

# Format effect sizes and zero percentages
results_display["Effect size (epsilon-squared)"] = results[
    "Effect size (epsilon-squared)"
].round(4)

results_display["% statements with zero"] = results[
    "% statements with zero"
].round(2)

display(results_display)


,Feature,Kruskal-Wallis H,P-value,Effect size (epsilon-squared),% statements with zero,Adjusted p-value
0,Past-tense verbs,4017.872590,0.000e+00,0.0762,43.95,0.000e+00
1,First-person singular pronouns,10170.835054,0.000e+00,0.1930,21.50,0.000e+00
2,Absolutist language,10472.732987,0.000e+00,0.1987,47.61,0.000e+00


In [14]:
abn = data[data["status"] != "Normal"]

results_abn = []
for feature_name, col in features.items():
    groups = [g[col].dropna().to_numpy() for _, g in abn.groupby("status")]
    H, p = kruskal(*groups)
    n = sum(len(g) for g in groups); k = len(groups)
    results_abn.append({
        "Feature": feature_name,
        "H": H,
        "P-value": p,
        "Effect size (epsilon-squared)": max(0, (H - k + 1) / (n - k)),
    })

results_abn = pd.DataFrame(results_abn)
results_abn["Adjusted p-value"] = multipletests(results_abn["P-value"], method="fdr_bh")[1]
display(results_abn.round(5))

,Feature,H,P-value,Effect size (epsilon-squared),Adjusted p-value
0,Past-tense verbs,491.19963,0.0,0.01338,0.0
1,First-person singular pronouns,2042.08157,0.0,0.05607,0.0
2,Absolutist language,473.76696,0.0,0.01290,0.0
